# Fast tokenizers' special powers (PyTorch)
### What this notebook teaches you:
1. Fast tokenizers return a special `BatchEncoding` object (not just a plain dict)
2. They track *where* each token came from in the original string (offset mapping)
3. We'll use those offsets to manually rebuild what the NER pipeline does internally

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
# Install required libraries — run this once, then restart your runtime
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer

# Load the BERT tokenizer. 'bert-base-cased' means:
#   - bert-base  → smaller/faster version of BERT
#   - cased      → it treats 'Hello' and 'hello' as different tokens
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

example = "My name is Sylvain and I work at Hugging Face in Brooklyn."

# Tokenize the sentence. This returns a BatchEncoding object,
# NOT a plain dict — it has extra methods like .tokens(), .word_ids(), etc.
encoding = tokenizer(example)

# Should print: <class 'transformers.tokenization_utils_base.BatchEncoding'>
print(type(encoding))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [3]:
# Check if the TOKENIZER itself is a fast tokenizer (backed by Rust, not Python)
# Fast tokenizers are much quicker on large batches AND track offsets automatically
# Should print: True
tokenizer.is_fast

True

In [4]:
# Check if the ENCODING (the result) also knows it came from a fast tokenizer
# Same info, different place to check it
# Should print: True
encoding.is_fast

True

In [5]:
# .tokens() gives you the actual string tokens — no need to decode IDs back
# Notice:
#   [CLS] and [SEP] are special BERT bookend tokens (added automatically)
#   'Sylvain' → split into S, ##yl, ##va, ##in  (## means 'continuation of prev token')
#   'Hugging' → split into Hu, ##gging
encoding.tokens()
# Output: ['[CLS]', 'My', 'name', 'is', 'S', '##yl', '##va', '##in', 'and', 'I',
#           'work', 'at', 'Hu', '##gging', 'Face', 'in', 'Brooklyn', '.', '[SEP]']

['[CLS]',
 'My',
 'name',
 'is',
 'S',
 '##yl',
 '##va',
 '##in',
 'and',
 'I',
 'work',
 'at',
 'Hu',
 '##gging',
 'Face',
 'in',
 'Brooklyn',
 '.',
 '[SEP]']

In [6]:
# .word_ids() maps each TOKEN back to which WORD it came from
# 'None' = special tokens ([CLS], [SEP]) — they don't belong to any word
# '3'    = word index 3 in the sentence, which is 'Sylvain'
#           Note: S, ##yl, ##va, ##in all have word_id=3 (they're all parts of one word)
# '8'    = 'Hugging' — Hu and ##gging both map to word 8
encoding.word_ids()
# Output: [None, 0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]

[None, 0, 1, 2, 3, 3, 3, 3, 4, 5, 6, 7, 8, 8, 9, 10, 11, 12, None]

In [7]:
# .word_to_chars(n) gives the character start/end positions of word #n
# Word index 3 is 'Sylvain' — let's prove it by slicing the original string
start, end = encoding.word_to_chars(3)
example[start:end]  # Should print: 'Sylvain'

'Sylvain'

---
## Part 2: NER (Named Entity Recognition) Pipeline
Now we use a real NER model. NER = find which words are people, places, organisations, etc.
First we'll use the black-box `pipeline()`, then we'll replicate it manually step by step.

In [8]:
from transformers import pipeline

# pipeline('token-classification') is the shorthand for NER
# It runs tokenize → model → decode all in one line
# Each result = one token with its entity label and confidence score
# 'I-PER' = inside a person entity, 'I-ORG' = inside an org, 'I-LOC' = inside a location
token_classifier = pipeline("token-classification")
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
# You'll see 'S', '##yl', '##va', '##in' all labeled I-PER (they're all parts of 'Sylvain')

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'entity': 'I-PER',
  'score': np.float32(0.99938285),
  'index': 4,
  'word': 'S',
  'start': 11,
  'end': 12},
 {'entity': 'I-PER',
  'score': np.float32(0.99815494),
  'index': 5,
  'word': '##yl',
  'start': 12,
  'end': 14},
 {'entity': 'I-PER',
  'score': np.float32(0.99590707),
  'index': 6,
  'word': '##va',
  'start': 14,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.99923277),
  'index': 7,
  'word': '##in',
  'start': 16,
  'end': 18},
 {'entity': 'I-ORG',
  'score': np.float32(0.9738931),
  'index': 12,
  'word': 'Hu',
  'start': 33,
  'end': 35},
 {'entity': 'I-ORG',
  'score': np.float32(0.976115),
  'index': 13,
  'word': '##gging',
  'start': 35,
  'end': 40},
 {'entity': 'I-ORG',
  'score': np.float32(0.9887976),
  'index': 14,
  'word': 'Face',
  'start': 41,
  'end': 45},
 {'entity': 'I-LOC',
  'score': np.float32(0.9932106),
  'index': 16,
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [9]:
from transformers import pipeline

# aggregation_strategy='simple' → merge subword tokens into full words automatically
# So instead of [S, ##yl, ##va, ##in] → you get one clean result: 'Sylvain'
# Score becomes the AVERAGE of all the merged tokens' scores
token_classifier = pipeline("token-classification", aggregation_strategy="simple")
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
# Output: [{'entity_group': 'PER', 'word': 'Sylvain', ...},
#           {'entity_group': 'ORG', 'word': 'Hugging Face', ...},
#           {'entity_group': 'LOC', 'word': 'Brooklyn', ...}]

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

---
## Part 3: Doing it manually (no pipeline shortcut)
Same result, but we control every step: tokenize → model forward pass → softmax → decode labels

In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# This is a BERT model fine-tuned specifically for NER on the CoNLL-2003 dataset
# 'cased' = treats capitalisation as meaningful (important for names!)
model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is Sylvain and I work at Hugging Face in Brooklyn."

# return_tensors='pt' → return PyTorch tensors (not plain lists)
# This is required to pass the input into the model
inputs = tokenizer(example, return_tensors="pt")

# Forward pass: feed the token IDs through the model
# outputs.logits = raw scores (one score per label, per token) — NOT probabilities yet
outputs = model(**inputs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Check the shapes to understand what we're working with
print(inputs["input_ids"].shape)  # [1, 19] → 1 sentence, 19 tokens
print(outputs.logits.shape)       # [1, 19, 9] → 1 sentence, 19 tokens, 9 possible labels
# The 9 labels are: O, B-MISC, I-MISC, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC

torch.Size([1, 19])
torch.Size([1, 19, 9])


In [12]:
import torch

# softmax converts raw logits → probabilities (all values 0-1 that sum to 1)
# dim=-1 means apply softmax across the last dimension (the 9 label scores per token)
# [0] selects the first (only) item in our batch
# .tolist() converts from tensor → plain Python list
probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()

# argmax picks the index of the highest probability for each token = the predicted label
# predictions is now a list of 19 integers like [0, 0, 0, 0, 4, 4, 4, 4, ...]
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)
# 0 = 'O' (outside/no entity), 4 = 'I-PER', 6 = 'I-ORG', 8 = 'I-LOC'

[0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 6, 6, 6, 0, 8, 0, 0]


In [13]:
# model.config.id2label is a dict that converts prediction integers → human-readable labels
# B- = Beginning of an entity, I- = Inside (continuation of) an entity
# O  = Outside (not part of any entity)
model.config.id2label
# {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER',
#  5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}

{0: 'O',
 1: 'B-MISC',
 2: 'I-MISC',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

In [14]:
# Build results WITHOUT offsets (v1 — basic version)
# We loop over every token's prediction, skip 'O' (no entity), keep the rest
results = []
tokens = inputs.tokens()  # Get the string tokens e.g. ['[CLS]', 'My', 'S', '##yl', ...]

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]  # e.g. pred=4 → 'I-PER'
    if label != "O":  # skip non-entity tokens
        results.append(
            {"entity": label,
             "score": probabilities[idx][pred],  # confidence for this label
             "word": tokens[idx]}                # e.g. '##yl'
        )

print(results)
# Problem: we get subwords ('S', '##yl', '##va', '##in') as separate entries
# and no start/end positions. That's what offsets fix.

[{'entity': 'I-PER', 'score': 0.9993828535079956, 'word': 'S'}, {'entity': 'I-PER', 'score': 0.9981548190116882, 'word': '##yl'}, {'entity': 'I-PER', 'score': 0.995907187461853, 'word': '##va'}, {'entity': 'I-PER', 'score': 0.9992327690124512, 'word': '##in'}, {'entity': 'I-ORG', 'score': 0.9738931059837341, 'word': 'Hu'}, {'entity': 'I-ORG', 'score': 0.9761149883270264, 'word': '##gging'}, {'entity': 'I-ORG', 'score': 0.9887974858283997, 'word': 'Face'}, {'entity': 'I-LOC', 'score': 0.99321049451828, 'word': 'Brooklyn'}]


In [15]:
# Ask the tokenizer to also return offset_mapping
# offset_mapping = a list of (start, end) character positions for each token
# This tells us WHERE in the original string each token came from
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets["offset_mapping"]
# e.g. (0,0) = [CLS] special token, (11,12) = 'S', (12,14) = '##yl'
# (33,35) = 'Hu', (35,40) = '##gging', (41,45) = 'Face'

[(0, 0),
 (0, 2),
 (3, 7),
 (8, 10),
 (11, 12),
 (12, 14),
 (14, 16),
 (16, 18),
 (19, 22),
 (23, 24),
 (25, 29),
 (30, 32),
 (33, 35),
 (35, 40),
 (41, 45),
 (46, 48),
 (49, 57),
 (57, 58),
 (0, 0)]

In [16]:
# Quick proof that offsets work:
# The offset for '##yl' is (12, 14), so slicing the original string gives us 'yl'
# (no ## prefix — that was added by the tokenizer, not in the real text)
example[12:14]  # Output: 'yl'

'yl'

In [17]:
# Build results WITH offsets (v2 — now includes start/end positions)
# This matches exactly what the pipeline() returns
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]  # list of (start, end) per token

for idx, pred in enumerate(predictions):
    label = model.config.id2label[pred]
    if label != "O":
        start, end = offsets[idx]  # character positions in the original string
        results.append(
            {
                "entity": label,
                "score": probabilities[idx][pred],
                "word": tokens[idx],
                "start": start,  # NEW: where this token starts in the original sentence
                "end": end,      # NEW: where it ends
            }
        )

print(results)  # Now matches the pipeline output exactly!

[{'entity': 'I-PER', 'score': 0.9993828535079956, 'word': 'S', 'start': 11, 'end': 12}, {'entity': 'I-PER', 'score': 0.9981548190116882, 'word': '##yl', 'start': 12, 'end': 14}, {'entity': 'I-PER', 'score': 0.995907187461853, 'word': '##va', 'start': 14, 'end': 16}, {'entity': 'I-PER', 'score': 0.9992327690124512, 'word': '##in', 'start': 16, 'end': 18}, {'entity': 'I-ORG', 'score': 0.9738931059837341, 'word': 'Hu', 'start': 33, 'end': 35}, {'entity': 'I-ORG', 'score': 0.9761149883270264, 'word': '##gging', 'start': 35, 'end': 40}, {'entity': 'I-ORG', 'score': 0.9887974858283997, 'word': 'Face', 'start': 41, 'end': 45}, {'entity': 'I-LOC', 'score': 0.99321049451828, 'word': 'Brooklyn', 'start': 49, 'end': 57}]


In [18]:
# Proof that offsets let us reconstruct multi-token entities cleanly
# 'Hu' starts at 33, 'Face' ends at 45 → slice the original string directly
# No need to hack around ## prefixes or join logic!
example[33:45]  # Output: 'Hugging Face'

'Hugging Face'

In [19]:
import numpy as np

# Build results WITH grouping (v3 — the full pipeline recreation)
# Groups consecutive I-XXX tokens into one entity, like the pipeline's aggregation_strategy
results = []
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
tokens = inputs_with_offsets.tokens()
offsets = inputs_with_offsets["offset_mapping"]

idx = 0
while idx < len(predictions):
    pred = predictions[idx]
    label = model.config.id2label[pred]  # e.g. 'I-PER'

    if label != "O":  # we found the start of an entity
        # Strip the 'B-' or 'I-' prefix to get just the entity type: 'PER', 'ORG', 'LOC'
        label = label[2:]
        start, _ = offsets[idx]  # record where this entity starts in the original string

        # Inner loop: keep consuming tokens as long as they're still I-{same_label}
        # This groups 'S', '##yl', '##va', '##in' → one 'Sylvain' entity
        all_scores = []
        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])  # collect each token's score
            _, end = offsets[idx]  # keep updating end position as we consume tokens
            idx += 1

        # Average all token scores → single confidence score for the whole entity
        score = np.mean(all_scores).item()

        # Use start/end offsets to slice the original string → no ## mess, clean word
        word = example[start:end]

        results.append(
            {
                "entity_group": label,  # 'PER', 'ORG', or 'LOC'
                "score": score,
                "word": word,           # e.g. 'Sylvain', 'Hugging Face', 'Brooklyn'
                "start": start,
                "end": end,
            }
        )
    idx += 1  # move to next token

print(results)
# Should match the pipeline output with aggregation_strategy='simple' exactly!

[{'entity_group': 'PER', 'score': 0.998169407248497, 'word': 'Sylvain', 'start': 11, 'end': 18}, {'entity_group': 'ORG', 'score': 0.9796018600463867, 'word': 'Hugging Face', 'start': 33, 'end': 45}, {'entity_group': 'LOC', 'score': 0.99321049451828, 'word': 'Brooklyn', 'start': 49, 'end': 57}]
